# Capítulo 1 — Análise Exploratória de Dados

Notebook com o **código** deste capítulo, pronto para o Google Colab. A explicação de cada trecho está no site do livro; aqui você roda os exemplos.

Rode a célula de **setup** abaixo primeiro (uma vez), depois as demais em ordem.

In [ ]:
# Setup (rode uma vez).
!pip install -q wquantiles
!curl -sO https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/formato.py   # baixa o ajudante de formatação do livro

## 1.1 — Elementos de Dados Estruturados

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)

In [ ]:
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")
estado.dtypes

In [ ]:
estado["Sigla"] = estado["Sigla"].astype("category")
print(estado["Sigla"].dtype)
print("categorias:", len(estado["Sigla"].cat.categories))

In [ ]:
from pandas.api.types import CategoricalDtype

tamanho = CategoricalDtype(categories=["pequeno", "medio", "grande"], ordered=True)
s = pd.Series(["grande", "pequeno", "medio"], dtype=tamanho)

print("ordenado:", s.sort_values().tolist())
print("maior que 'pequeno'?", s.gt("pequeno").tolist())
print("máximo:", s.max())

In [ ]:
nominal = pd.Series(["grande", "pequeno", "medio"], dtype="category")  # sem ordered=True

try:
    nominal.gt("pequeno")
except TypeError as erro:
    print("TypeError:", erro)

## 1.2 — Dados Retangulares

In [ ]:
import pandas as pd

In [ ]:
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")
print("linhas x colunas:", estado.shape)
estado.head()

In [ ]:
estado.info()

In [ ]:
print("índice padrão:", estado.index[:5].tolist())

# Um índice significativo torna a busca por rótulo natural
por_estado = estado.set_index("Sigla")
por_estado.loc["SP"]

## 1.3 — Estimativas de Localização

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import trim_mean
import wquantiles
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
media = estado["Populacao"].mean()
print(f"Média: {num(media)}")

In [ ]:
media_aparada = trim_mean(estado["Populacao"], 0.1)
print(f"Média aparada (10%): {num(media_aparada)}")

In [ ]:
mediana = estado["Populacao"].median()
print(f"Mediana: {num(mediana)}")

# Qual estado é a mediana? Com n ímpar, ela é uma observação de verdade.
estado_mediano = estado.loc[estado["Populacao"] == mediana, "Estado"].iloc[0]
print(f"É a população de: {estado_mediano}")

In [ ]:
print(f"Valores únicos de população: {estado['Populacao'].nunique()} de {len(estado)}")

In [ ]:
dfw = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/dfw_airline.csv").rename(columns={
    "Carrier": "Companhia", "ATC": "ControleAereo", "Weather": "Clima",
    "Security": "Seguranca", "Inbound": "VooAnterior"})
print(f"Causa modal de atraso: {dfw.iloc[0].idxmax()}")

In [ ]:
media_pond = np.average(estado["Taxa.Homicidios"], weights=estado["Populacao"])
mediana_pond = wquantiles.median(estado["Taxa.Homicidios"], weights=estado["Populacao"])

print(f"Média simples     : {num(estado['Taxa.Homicidios'].mean(), 4)}")
print(f"Média ponderada   : {num(media_pond, 4)}")
print(f"Mediana ponderada : {num(mediana_pond, 4)}")

In [ ]:
fig, ax = plt.subplots()

ax.hist(estado["Populacao"] / 1e6, bins=20, color="#b0c4d8", edgecolor="white")
ax.axvline(media / 1e6, color="#c0392b", linestyle="-", linewidth=2, label=f"Média: {num(media/1e6, 1)}M")
ax.axvline(media_aparada / 1e6, color="#e67e22", linestyle="--", linewidth=2, label=f"Média aparada: {num(media_aparada/1e6, 1)}M")
ax.axvline(mediana / 1e6, color="#27ae60", linestyle=":", linewidth=2.5, label=f"Mediana: {num(mediana/1e6, 1)}M")

ax.set_xlabel("População (milhões)")
ax.set_ylabel("Número de estados")
ax.legend()
plt.tight_layout()
plt.show()

## 1.4 — Estimativas de Variabilidade

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels import robust
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")

In [ ]:
desvio = estado["Populacao"].std()
print(f"Desvio-padrão: {num(desvio)}")

In [ ]:
iqr = estado["Populacao"].quantile(0.75) - estado["Populacao"].quantile(0.25)
print(f"IQR: {num(iqr)}")

In [ ]:
mad = robust.scale.mad(estado["Populacao"])
print(f"MAD: {num(mad)}")

# O mesmo cálculo, explicitamente — para mostrar de onde vem o 0,6745
mad_manual = abs(estado["Populacao"] - estado["Populacao"].median()).median() / 0.6744897501960817
print(f"MAD (manual): {num(mad_manual)}")

In [ ]:
mediana = estado["Populacao"].median()
fig, ax = plt.subplots()

ax.hist(estado["Populacao"] / 1e6, bins=20, color="#b0c4d8", edgecolor="white")
ax.axvline(mediana / 1e6, color="#2c3e50", linewidth=2, label=f"Mediana: {num(mediana/1e6, 1)}M")

for medida, valor, cor, estilo in [
    ("Desvio-padrão", desvio, "#c0392b", "-"),
    ("IQR", iqr, "#e67e22", "--"),
    ("MAD", mad, "#27ae60", ":"),
]:
    ax.axvline((mediana + valor) / 1e6, color=cor, linestyle=estilo, linewidth=2,
               label=f"{medida}: {num(valor/1e6, 1)}M")

ax.set_xlabel("População (milhões)")
ax.set_ylabel("Número de estados")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
mu, sd = estado["Populacao"].mean(), estado["Populacao"].std(ddof=1)
sp = estado.loc[estado["Sigla"] == "SP", "Populacao"].iloc[0]
z_sp = (sp - mu) / sd
print(f"z-score da população de SP: {num(z_sp, 2)}")

mu_t, sd_t = estado["Taxa.Homicidios"].mean(), estado["Taxa.Homicidios"].std(ddof=1)
sp_t = estado.loc[estado["Sigla"] == "SP", "Taxa.Homicidios"].iloc[0]
print(f"z-score da taxa de homicídios de SP: {num((sp_t - mu_t) / sd_t, 2)}")

## 1.5 — Explorando a Distribuição dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")

In [ ]:
percentis = [0.05, 0.25, 0.5, 0.75, 0.95]
tabela = pd.DataFrame(estado["Taxa.Homicidios"].quantile(percentis))
tabela.index = [f"{p:.0%}" for p in percentis]
tabela.transpose().map(lambda v: num(v, 3))

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5))
(estado["Populacao"] / 1e6).plot.box(ax=ax)
ax.set_ylabel("População (milhões)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5))
ax.violinplot(estado["Populacao"] / 1e6, showmedians=True)
ax.set_ylabel("População (milhões)")
ax.set_xticks([])
plt.tight_layout()
plt.show()

In [ ]:
faixas = pd.cut(estado["Populacao"], 10)
faixas.value_counts().sort_index()

In [ ]:
fig, ax = plt.subplots()
(estado["Populacao"] / 1e6).plot.hist(ax=ax, bins=10, edgecolor="white")
ax.set_xlabel("População (milhões)")
ax.set_ylabel("Número de estados")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
estado["Taxa.Homicidios"].plot.hist(ax=ax, density=True, xlim=[0, 40],
                                bins=range(0, 40, 2), edgecolor="white")
estado["Taxa.Homicidios"].plot.density(ax=ax, linewidth=2)
ax.set_xlabel("Taxa de homicídios (por 100.000)")
plt.tight_layout()
plt.show()

In [ ]:
N, MEDIA, DESVIO = 25, 65.0, 12.0

def padroniza(x, media=MEDIA, desvio=DESVIO):
    """Crava média e desvio exatos, via z-score e reescala.

    A assimetria é invariante a transformação linear — ela sobrevive intacta.
    É isso que permite três conjuntos com a MESMA média e a MESMA variância,
    mas formas diferentes.
    """
    x = np.asarray(x, float)
    return (x - x.mean()) / x.std(ddof=1) * desvio + media

# Um gerador por turma: a ordem de consumo do RNG não pode alterar um conjunto
# em silêncio quando outro mudar.
turma_a = padroniza(np.random.default_rng(42).lognormal(0, 1.0, N))

# Espelho em torno da média: preserva média e variância, e inverte o SINAL da
# assimetria — exatamente, não aproximadamente.
turma_b = 2 * MEDIA - turma_a

# Simétrica POR CONSTRUÇÃO, não por amostragem: 12 desvios, seus 12 espelhos,
# e o centro. A assimetria é exatamente zero, não "próxima de zero".
desvios = np.abs(np.random.default_rng(7).normal(0, 1, (N - 1) // 2))
turma_c = padroniza(np.concatenate([-desvios, [0.0], desvios]))

turmas = {"A — à direita": turma_a, "B — à esquerda": turma_b, "C — simétrica": turma_c}

# As garantias são verificadas, não afirmadas.
assert np.allclose([t.mean() for t in turmas.values()], MEDIA)
assert np.allclose([t.var(ddof=1) for t in turmas.values()], DESVIO ** 2)
assert np.isclose(stats.skew(turma_a, bias=False), -stats.skew(turma_b, bias=False))
assert np.isclose(stats.skew(turma_c, bias=False), 0, atol=1e-12)
assert all(t.min() >= 0 and t.max() <= 100 for t in turmas.values())

In [ ]:
resumo = pd.DataFrame(
    {
        "Média": [num(t.mean(), 1) for t in turmas.values()],
        "Mediana": [num(np.median(t), 1) for t in turmas.values()],
        "Variância": [num(t.var(ddof=1), 1) for t in turmas.values()],
        "Assimetria": [num(stats.skew(t, bias=False), 2) for t in turmas.values()],
    },
    index=list(turmas),
)
resumo

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(11, 3.3), sharey=True)

for ax, (nome, notas) in zip(eixos, turmas.items()):
    ax.hist(notas, bins=np.arange(30, 101, 7), color="#b0c4d8", edgecolor="white")
    ax.axvline(notas.mean(), color="#c0392b", linewidth=2)
    ax.axvline(np.median(notas), color="#27ae60", linestyle="--", linewidth=2)
    ax.set_title(nome, fontsize=10)
    ax.set_xlabel("Nota")

eixos[0].set_ylabel("Alunos")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

caixas = ax.boxplot(
    [turma_c, turma_b, turma_a],          # de baixo para cima: simétrica, esquerda, direita
    tick_labels=["C — simétrica", "B — à esquerda", "A — à direita"],
    vert=False,
    patch_artist=True,
    medianprops={"color": "#27ae60", "linewidth": 2},
    widths=0.6,
)
for caixa in caixas["boxes"]:
    caixa.set(facecolor="#b0c4d8", edgecolor="#2c3e50")

ax.set_xlabel("Nota")
plt.tight_layout()
plt.show()

In [ ]:
print(f"População das UFs  : {num(stats.skew(estado['Populacao'], bias=False), 2)}")
print(f"Taxa de homicídios : {num(stats.skew(estado['Taxa.Homicidios'], bias=False), 2)}")

In [ ]:
print(f"Curtose da população das UFs : {num(stats.kurtosis(estado['Populacao']), 2)}")
print(f"Curtose da taxa de homicídios: {num(stats.kurtosis(estado['Taxa.Homicidios']), 2)}")

## 1.6 — Explorando Dados Binários e Categóricos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
dfw = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/dfw_airline.csv").rename(columns={
    "Carrier": "Companhia",
    "ATC": "ControleAereo",
    "Weather": "Clima",
    "Security": "Seguranca",
    "Inbound": "VooAnterior",
})
proporcoes = 100 * dfw / dfw.values.sum()
proporcoes.round(2).map(lambda v: num(v, 2))

In [ ]:
fig, ax = plt.subplots()
dfw.transpose().plot.bar(ax=ax, legend=False, color="#4a90a4", edgecolor="white")
ax.set_xlabel("Causa do atraso")
ax.set_ylabel("Número de atrasos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")
nordeste = ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"]
ne = estado[estado["Sigla"].isin(nordeste)].sort_values("Sigla")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(ne["Sigla"], ne["Taxa.Homicidios"], color="#b0c4d8", edgecolor="white")
ax.set_ylim(20, 38)
ax.set_xlabel("Estado")
ax.set_ylabel("Taxa de homicídios (por 100 mil)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(ne["Sigla"], ne["Taxa.Homicidios"], color="#b0c4d8", edgecolor="white")
ax.set_ylim(0, 38)
ax.set_xlabel("Estado")
ax.set_ylabel("Taxa de homicídios (por 100 mil)")
plt.tight_layout()
plt.show()

In [ ]:
moda = proporcoes.transpose().iloc[:, 0].idxmax()
print(f"Moda (causa mais frequente): {moda}")

In [ ]:
# Uma companhia estima o custo médio de compensação por passageiro,
# conforme a causa do atraso.
custo = {"Companhia": 180, "ControleAereo": 40, "Clima": 0, "Seguranca": 25, "VooAnterior": 120}

p = (dfw / dfw.values.sum()).transpose().iloc[:, 0]   # probabilidade de cada causa
ve = sum(p[causa] * valor for causa, valor in custo.items())

print(f"Valor esperado do custo por atraso: R$ {num(ve, 2)}")

## 1.7 — Correlação

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
SETORES = {
    "consumer_discretionary": "consumo_discricionario",
    "consumer_staples": "consumo_essencial",
    "energy": "energia",
    "etf": "etf",
    "financials": "financeiro",
    "health_care": "saude",
    "industrials": "industrial",
    "information_technology": "tecnologia_da_informacao",
    "materials": "materiais",
    "telecommunications_services": "telecomunicacoes",
    "utilities": "utilidade_publica",
}
setores = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/sp500_sectors.csv").rename(
    columns={"sector": "Setor", "symbol": "Simbolo"}
)
setores["Setor"] = setores["Setor"].replace(SETORES)
precos  = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/sp500_data.csv.gz", index_col=0)

simbolos_telecom = setores[setores["Setor"] == "telecomunicacoes"]["Simbolo"]
telecom = precos.loc[precos.index >= "2012-07-01", simbolos_telecom]

print("dias x empresas:", telecom.shape)
telecom.head()

In [ ]:
telecom.corr().round(3).map(lambda v: num(v, 3))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(telecom["T"], telecom["VZ"], alpha=0.5, s=20, color="#2c7fb8")
ax.axhline(0, color="grey", linewidth=0.8)
ax.axvline(0, color="grey", linewidth=0.8)
ax.set_xlabel("Retorno diário — AT&T (T)")
ax.set_ylabel("Retorno diário — Verizon (VZ)")
plt.tight_layout()
plt.show()

In [ ]:
etfs = precos.loc[precos.index > "2012-07-01",
                  setores[setores["Setor"] == "etf"]["Simbolo"]]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(etfs.corr(), vmin=-1, vmax=1,
            cmap=sns.diverging_palette(20, 220, as_cmap=True), ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
x = np.linspace(-1, 1, 200)
y = x ** 2

print(f"Correlação entre x e x²: {np.corrcoef(x, y)[0, 1]:.3e}")

## 1.8 — Explorando Duas ou Mais Variáveis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
kc = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/kc_tax.csv.gz").rename(columns={
    "TaxAssessedValue": "ValorVenal",
    "SqFtTotLiving": "AreaConstruida",
    "ZipCode": "CEP",
})
print(f"registros brutos: {num(len(kc), 0)}")

kc0 = kc.loc[(kc.ValorVenal < 750000) &
             (kc.AreaConstruida > 100) &
             (kc.AreaConstruida < 3500), :]
print(f"após filtrar extremos: {num(len(kc0), 0)}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
kc0.plot.hexbin(x="AreaConstruida", y="ValorVenal",
                gridsize=30, sharex=False, ax=ax)
ax.set_xlabel("Área construída (pés²)")
ax.set_ylabel("Valor venal (US$)")
plt.tight_layout()
plt.show()

In [ ]:
amostra = kc0.sample(10000, random_state=42)

fig, ax = plt.subplots(figsize=(6, 5))
sns.kdeplot(data=amostra, x="AreaConstruida", y="ValorVenal", ax=ax)
ax.set_xlabel("Área construída (pés²)")
ax.set_ylabel("Valor venal (US$)")
plt.tight_layout()
plt.show()

In [ ]:
SITUACAO = {
    "Fully Paid": "Quitado",
    "Current": "Em dia",
    "Late": "Atrasado",
    "Charged Off": "Inadimplente",
}
lc = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/lc_loans.csv").rename(columns={"status": "Situacao", "grade": "Nota"})
lc["Situacao"] = lc["Situacao"].replace(SITUACAO)

contagem = lc.pivot_table(index="Nota", columns="Situacao",
                          aggfunc=lambda x: len(x), margins=True)
contagem

In [ ]:
prop = contagem.copy().loc["A":"G", :].astype(float)
situacoes = [c for c in prop.columns if c != "All"]
prop.loc[:, situacoes] = prop.loc[:, situacoes].div(prop["All"], axis=0)
prop["All"] = prop["All"] / sum(prop["All"])
prop.round(3).map(lambda v: num(v, 3))

In [ ]:
voos = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/airline_stats.csv").rename(columns={
    "pct_carrier_delay": "pct_atraso_companhia",
    "pct_atc_delay": "pct_atraso_controle",
    "pct_weather_delay": "pct_atraso_clima",
    "airline": "Companhia",
})

fig, ax = plt.subplots(figsize=(7, 5))
voos.boxplot(by="Companhia", column="pct_atraso_companhia", ax=ax)
ax.set_xlabel("")
ax.set_ylabel("% diário de voos atrasados")
ax.set_ylim(0, 50)
plt.suptitle("")
plt.title("")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.violinplot(data=voos, x="Companhia", y="pct_atraso_companhia",
               ax=ax, inner="quartile", color="#b0c4d8")
ax.set_xlabel("")
ax.set_ylabel("% diário de voos atrasados")
ax.set_ylim(0, 50)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
kc_ceps = kc0.loc[kc0.CEP.isin([98188, 98105, 98108, 98126]), :]

def hexbin(x, y, color, **kwargs):
    cmap = sns.light_palette(color, as_cmap=True)
    plt.hexbin(x, y, gridsize=25, cmap=cmap, **kwargs)

g = sns.FacetGrid(kc_ceps, col="CEP", col_wrap=2, height=3.2)
g.map(hexbin, "AreaConstruida", "ValorVenal", extent=[0, 3500, 0, 700000])
g.set_axis_labels("Área construída (pés²)", "Valor venal (US$)")
plt.tight_layout()
plt.show()